In [1]:
!pip install transformers
!pip install "transformers[torch]"


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from transformers import (
    T5Tokenizer,
    Trainer,
    TrainingArguments,
    T5ForConditionalGeneration
)

In [3]:
train_data=pd.read_csv(f"data/samsum-train.csv")
vals_data=pd.read_csv(f"data/samsum-validation.csv")

In [4]:
train_data.head()


,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [5]:
train_data.shape

(14732, 3)

In [6]:
train_data['dialogue'][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [7]:
vals_data.shape

(818, 3)

In [8]:
train_data.sample(10)

,id,dialogue,summary
11385,13818397,"Evelyn: Pete I'll be late, pls tell that Mr. D...",Evelyn wants Pete to inform Mr. Dickinson she ...
9192,13809860,Vicky: You coming out tonight?\r\nDavid: Absol...,David and Vicky are meeting at Queen's Head at...
1927,13728529,Kylie: are you home? the delivery guy has got ...,The delivery guy has a free of charge package ...
803,13728661,Sam: I am completely baffled by recent events\...,"There is no solution to populism, tabloidisati..."
14549,13820346,Zachary: what did you choose?\r\nDale: I think...,"Zachary, Dalton and Dale are going to a barbecue."
4603,13862336,Jodie: Hi mummy\nMom: Hi sweetheart\nJodie: We...,Jodie went to the park with her child. Jodie s...
12056,13828407,"Patricia: did he call you?\r\nDerek: no, not y...",Derek and Patricia will wait for the call anot...
4209,13681730,Sandra: Are you sleeping?\r\nMatija: Nope\r\nS...,Matija is in his bed on Friday nights. Tomorro...
14067,13863109,Lucas: r u better now?\nKeira: yes much better...,Keira has been sick and couldn't work. She sta...
11413,13727635,"Lottie: Hi, you off to that Mums, Dads, Guardi...",Lottie and Naomi are going to the meeting afte...


### Ramdom Sampleing

In [9]:
train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data=vals_data.sample(n=500,random_state=42).reset_index(drop=True)

In [10]:
val_data.head()

,id,dialogue,summary
0,13680857,"Edd: wow, did you hear that they're transferri...",Rose and Edd will be transferred to a new depa...
1,13716124,"Tom: Where is the ""Sala del Capitolo""\r\nKevin...","""Sala del Capitolo"" Tom is looking for is in t..."
2,13864418,Patricia: The rowing practice is cancelled!\nK...,The rowing practice is cancelled. A few member...
3,13729340,"Tom: U OK?\r\nAlex: Yeah, pretty good. U?\r\nT...",Tom and Alex had fun last night. They drank a ...
4,13818813,"Patricia: Hello, here's the fair-trade brand I...",Patricia recommends a fair-trade brand she tal...


In [11]:
print(train_data.shape)
print(val_data.shape)

(4000, 3)
(500, 3)


### Pre-Processing

In [12]:
import re # Regular Expression

def clean_data(text):
    text = re.sub(r'\s+', ' ', text) #space
    text = re.sub(r'\r\n', '', text) #line
    text = re.sub(r'<.*?>', '', text) # Remove HTMl Tags
    text = text.strip().lower() #Remove extra space and convert into lower case
    return text

In [13]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)
val_data["dialogue"]=train_data["dialogue"].apply(clean_data)
val_data["summary"]=train_data["summary"].apply(clean_data)

In [14]:
train_data.sample(10)

,id,dialogue,summary
1325,13814058-1,"barbara: hey, could you buy some eggs? agatha:...",agatha is at tesco's and will buy eggs and muf...
3164,13862465,zoli: hi! zoli: please ping me when you are ar...,zoli wants to ask a request of jess. jess is o...
755,13821295,vlad: 2-3 vlad: still carson: flyers are going...,flyers are playing against the canucks.
3713,13612143,tyler: good morning! ashley: hello my dear tyl...,tyler is seeing ashley for drinks today. ashle...
2687,13716282,"camilla: good afternoon, ladies. as always, i’...","""a mercy"" by toni morrison is going to be the ..."
19,13727943,rick: how’s everything? got any plans for the ...,rick invited martin for his new year's eve par...
2813,13680890,stan: josh josh: sup stan: are you at home rig...,josh was looking for stan's red folder on stan...
2197,13818473,clarie: is that offer still available? aaron: ...,the home teaching offer is still available. cl...
2985,13717005,raelyn: anything to watch? truman: movie or se...,raelyn wants to watch a movie. tanner suggests...
473,13680777,ginger: hi ginger: can you help me? willow: of...,ginger wants to borrow willow's falsies but wi...


In [15]:
val_data.sample(10)

,id,dialogue,summary
136,13864439,mom: i've got the suitcase lee: thanks! lee: i...,lee will stop by mom's to pick up a suitcase. ...
104,13864653,terry: i need your email addresses to send pre...,"terry will send joe, jake and mindy a presenta..."
115,13680226,"william: emma’s christmas cake, decoration all...",emma made a cake decoration with santa claus w...
298,13682090,kai: should we meet at the railway station? fa...,faith will meet kai 4.30 at the railway statio...
122,13820786,rachel: hey how did it go mary: no idea... it ...,the results will be announced on monday morning.
109,13820899,rosie: have you sent the project to mr. smith?...,mike is working on a project for mr. smith. it...
247,13820929,nat: would you care to tell me what this is? n...,nat found a g-string under the bed. paul has n...
163,13728164,"watson: hey bella, please tell me some intervi...",watson has an interview tomorrow and wants to ...
410,13820652,jade: hi :-) could you please send me pics of ...,jade didn't come to the class because he had a...
148,13682540,tom: icant believe its already december igor: ...,"tom, igor and eric feel time is flying by."
